# California Housing Market : Feature engineering and feature selection
In the previous exercise, we concluded it was worth including more variables in a model. But is this set of variables **the best** we could have chosen ? In this exercises, we'll go further by applying two canonical methods:
* Feature engineering consists in creating more variables from the original dataset
* Feature selection allows to select the best set of features among all the available variables

## The dataset
1. Load the California Housing dataset again and remove the outliers:

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn import datasets

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [3]:
from sklearn import datasets
data = datasets.fetch_california_housing(data_home=None, download_if_missing=True, return_X_y=False)
data

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]]),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894]),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': '.. _california_housing_dataset:\n

In [4]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Price'] = data.target
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [5]:
df.shape

(20640, 9)

In [9]:
mask = (df['AveRooms'] < 10) & (df['AveBedrms'] < 10) & (df['Population'] < 15000) & (df['AveOccup'] < 10) & (df['Price'] < 5)
df = df.loc[mask,:]
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [10]:
df.shape

(19398, 9)

In [11]:
df.describe(include='all')

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
count,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000
mean,3.674497,28.496907,5.210648,1.066038,1442.172080,2.944640,35.637872,-119.567484,1.924128
std,1.563397,12.477953,1.168098,0.128846,1077.498768,0.766194,2.142960,2.004793,0.971784
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.750000,32.540000,-124.350000,0.149990
25%,2.525900,18.000000,4.407329,1.005413,805.000000,2.450413,33.930000,-121.770000,1.167000
50%,3.447800,29.000000,5.170038,1.047619,1185.500000,2.842105,34.260000,-118.490000,1.741000
75%,4.583175,37.000000,5.944617,1.096884,1752.000000,3.308127,37.720000,-118.000000,2.485000
max,15.000100,52.000000,9.979167,3.411111,13251.000000,9.954545,41.950000,-114.550000,4.991000


In [12]:
(df.isnull().sum()/len(df))*100

MedInc        0.0
HouseAge      0.0
AveRooms      0.0
AveBedrms     0.0
Population    0.0
AveOccup      0.0
Latitude      0.0
Longitude     0.0
Price         0.0
dtype: float64

2. Separate the target from the features

In [13]:
target_variable = "Price"

X = df.drop(target_variable, axis = 1)
Y = df.loc[:,target_variable]

print("Y:")
print(Y.loc[:5])
print()
print("X:")
X.head()

Y:
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
5    2.697
Name: Price, dtype: float64

X:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


## From linear to non-linear regression
An easy way of implementing a non-linear regression is to create by hand more columns containing non-linear functions of the features.

3. For each explanatory variable, create 3 new columns in $X$ containing the following functions:
* $\textrm{X}^2$
* $\textrm{X}^3$
* $\textrm{X}^4$
* $\frac{1}{\textrm{X}}$
* $\frac{1}{\textrm{X}^2}$

In [14]:
features_list = X.columns
for c in features_list:
    X.loc[:, c + '_2'] = X[c]**2
    X.loc[:, c + '_3'] = X[c]**3
    X.loc[:, c + '_4'] = X[c]**4
    X.loc[:, c + '_inverse'] = 1/X[c]
    X.loc[:, c + '_inverse2'] = 1/(X[c]**2)
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_2,MedInc_3,...,Latitude_2,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,69.308955,577.010912,...,1434.8944,54353.799872,2.058922e+06,0.026399,0.000697,14940.1729,-1.826137e+06,2.232088e+08,-0.008181,0.000067
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,68.913242,572.076387,...,1433.3796,54267.751656,2.054577e+06,0.026413,0.000698,14937.7284,-1.825689e+06,2.231357e+08,-0.008182,0.000067
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,52.669855,382.246204,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14942.6176,-1.826586e+06,2.232818e+08,-0.008181,0.000067
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,31.844578,179.702136,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14945.0625,-1.827034e+06,2.233549e+08,-0.008180,0.000067
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,14.793254,56.897815,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14945.0625,-1.827034e+06,2.233549e+08,-0.008180,0.000067


4. Split your dataset into train (80%) and test (20%)

In [15]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

5. Apply the same preprocessing as in the previous exercise

In [16]:
X_train.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_2,MedInc_3,...,Latitude_2,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2
3235,2.3889,6.0,6.316614,1.294671,992.0,3.109718,36.09,-119.57,5.706843,13.633078,...,1302.4881,47006.795529,1.696475e+06,0.027709,0.000768,14296.9849,-1.709490e+06,2.044038e+08,-0.008363,0.000070
13981,3.4912,7.0,8.355308,1.554795,2933.0,2.511130,34.85,-117.46,12.188477,42.552412,...,1214.5225,42326.109125,1.475065e+06,0.028694,0.000823,13796.8516,-1.620578e+06,1.903531e+08,-0.008514,0.000072
9219,1.9464,36.0,4.975510,1.053061,639.0,2.608163,37.12,-120.27,3.788473,7.373884,...,1377.8944,51147.440128,1.898593e+06,0.026940,0.000726,14464.8729,-1.739690e+06,2.092325e+08,-0.008315,0.000069
10851,3.1667,22.0,3.803838,1.000000,1952.0,2.081023,33.66,-117.90,10.027989,31.755632,...,1132.9956,38136.631896,1.283679e+06,0.029709,0.000883,13900.4100,-1.638858e+06,1.932214e+08,-0.008482,0.000072
8888,4.2520,31.0,3.978296,1.039389,1985.0,1.595659,34.03,-118.49,18.079504,76.874051,...,1158.0409,39408.131827,1.341059e+06,0.029386,0.000864,14039.8801,-1.663585e+06,1.971182e+08,-0.008440,0.000071


In [17]:
X_test.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_2,MedInc_3,...,Latitude_2,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2
17333,5.2990,12.0,7.214932,1.047511,1200.0,2.714932,34.91,-120.44,28.079401,148.792746,...,1218.7081,42545.099771,1.485249e+06,0.028645,0.000821,14505.7936,-1.747078e+06,2.104180e+08,-0.008303,0.000069
1012,2.6667,44.0,4.541284,1.027523,277.0,2.541284,37.68,-121.77,7.111289,18.963674,...,1419.7824,53497.400832,2.015782e+06,0.026539,0.000704,14827.9329,-1.805597e+06,2.198676e+08,-0.008212,0.000067
5124,1.5521,30.0,3.850679,1.002262,1966.0,4.447964,33.99,-118.26,2.409014,3.739031,...,1155.3201,39269.330199,1.334765e+06,0.029420,0.000866,13985.4276,-1.653917e+06,1.955922e+08,-0.008456,0.000072
1845,6.3538,49.0,6.293886,1.017751,1148.0,2.264300,37.90,-122.28,40.370774,256.507827,...,1436.4100,54439.939000,2.063274e+06,0.026385,0.000696,14952.3984,-1.828379e+06,2.235742e+08,-0.008178,0.000067
4035,3.2154,20.0,4.133444,1.060181,7450.0,1.772122,34.17,-118.52,10.338797,33.243368,...,1167.5889,39896.512713,1.363264e+06,0.029265,0.000856,14046.9904,-1.664849e+06,1.973179e+08,-0.008437,0.000071


In [18]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train

array([[-0.81992708, -1.808924  ,  0.95532843, ..., -0.02225618,
         0.01946793, -0.02784954],
       [-0.11259428, -1.72879039,  2.7064652 , ..., -1.04157365,
        -1.05947081,  1.06288768],
       [-1.10387407,  0.59508421, -0.19661352, ...,  0.32805116,
         0.3690467 , -0.37708585],
       ...,
       [-0.84200116,  0.35468339, -1.35426383, ..., -0.97089867,
        -0.98149075,  0.98340385],
       [ 0.22166105, -0.76718711, -0.21715438, ..., -0.78587401,
        -0.77967042,  0.77816261],
       [-0.67843486,  0.75535142, -0.00620751, ..., -0.28832253,
        -0.25301648,  0.2457817 ]])

In [19]:
X_test = scaler.transform(X_test)
X_test

array([[ 1.04744947, -1.32812236,  1.72693875, ...,  0.41405428,
         0.45333115, -0.46098272],
       [-0.64166613,  1.23615306, -0.56959193, ...,  1.09957975,
         1.10461025, -1.10527128],
       [-1.35689169,  0.11428257, -1.16278792, ..., -0.66150072,
        -0.64586306,  0.64246175],
       ...,
       [ 0.49097755,  0.59508421,  0.44373766, ..., -0.7811056 ,
        -0.77451308,  0.77292676],
       [ 0.09800775,  0.43481699, -0.14626229, ..., -0.81445904,
        -0.81063283,  0.80960571],
       [ 1.06381252,  0.83548503,  1.12012678, ...,  0.44957334,
         0.48796739, -0.49542534]])

6. Train a model including all these features. Do you get better performances than before?

In [20]:
regressor = LinearRegression()
regressor.fit(X_train, Y_train)

LinearRegression()

In [21]:
print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

R2 score on training set :  0.6831497561839482
R2 score on test set :  0.6858617124117808


## Forward selection
This feature engineering trick improved the model's score significantly ! But now, the model is a lot more complex as it uses 32 input features. Do we really need all these features? Let's implement the forward selection method described in this morning's lecture. 

Fortunately, the latest versions of sklearn provide a class that implements forward selection, such that we don't need to code the algorithm by hand 🥳

7. Have a look at the documentation of [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) and try to understand the following lines of code:

In [22]:
from sklearn.feature_selection import  SequentialFeatureSelector
feature_selector =  SequentialFeatureSelector(regressor, n_features_to_select = 20)
feature_selector.fit(X_train, Y_train)
features_list = X.columns
best_features = features_list[feature_selector.support_]
print("According to the forward selection algorithm, the following features should be kept: ")
print(best_features.to_list())

According to the forward selection algorithm, the following features should be kept: 
['MedInc', 'HouseAge', 'Population', 'Latitude', 'MedInc_inverse2', 'AveRooms_3', 'AveRooms_4', 'AveRooms_inverse', 'AveRooms_inverse2', 'AveBedrms_inverse', 'Population_2', 'Population_inverse2', 'AveOccup_3', 'AveOccup_inverse', 'AveOccup_inverse2', 'Latitude_3', 'Latitude_4', 'Latitude_inverse', 'Latitude_inverse2', 'Longitude_inverse']


8. Create a DataFrame X_best containing only the best set of features, train a model only with these features and evaluate the performances

In [23]:
X_best = X.loc[:, best_features]
X_train, X_test, Y_train, Y_test = train_test_split(X_best, Y, test_size=0.2, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

regressor = LinearRegression()
regressor.fit(X_train, Y_train)

print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

R2 score on training set :  0.6641141198225633
R2 score on test set :  0.6727171115609578


## Advanced feature engineering
Let's make even more advanced feature engineering. Until now, we've included the latitude and longitude as such into the models. However, usually the GPS coordinates are not used rawly, instead we deduce some geographical information from these. Let's use an API that will allows to retrieve the name of the city from the latitude and longitude.

💡 As the calls to the API may be time-consuming, we'll work on a sample of the dataset.

9. Take a sample of your dataset X (the one that contains all the features and not only the best set, because we need the values of Latitude and Longitude). Keep only 150 rows.

In [24]:
X_sample = X.sample(150)
X_sample.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_2,MedInc_3,...,Latitude_2,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2
12593,2.3487,38.0,4.267647,0.926471,786.0,2.311765,38.53,-121.48,5.516392,12.956349,...,1484.5609,57200.131477,2.203921e+06,0.025954,0.000674,14757.3904,-1.792728e+06,2.177806e+08,-0.008232,0.000068
11915,3.1384,46.0,5.627249,1.087404,866.0,2.226221,33.95,-117.40,9.849555,30.911842,...,1152.6025,39130.854875,1.328493e+06,0.029455,0.000868,13782.7600,-1.618096e+06,1.899645e+08,-0.008518,0.000073
13036,3.0000,37.0,4.890625,1.042969,686.0,2.679688,38.68,-121.17,9.000000,27.000000,...,1496.1424,57870.788032,2.238442e+06,0.025853,0.000668,14682.1689,-1.779038e+06,2.155661e+08,-0.008253,0.000068
6824,2.5430,37.0,4.550314,1.036164,1977.0,3.108491,34.08,-118.10,6.466849,16.445197,...,1161.4464,39582.093312,1.348958e+06,0.029343,0.000861,13947.6100,-1.647213e+06,1.945358e+08,-0.008467,0.000072
13563,3.8125,38.0,5.758721,0.997093,796.0,2.313953,34.15,-117.28,14.535156,55.415283,...,1166.2225,39826.498375,1.360075e+06,0.029283,0.000857,13754.5984,-1.613139e+06,1.891890e+08,-0.008527,0.000073


10. Create a Y_sample variable containing the target values corresponding to the rows that were kept in X_sample

In [25]:
Y_sample = Y.sample(150)
Y_sample.head()

3274     0.935
10221    1.750
13913    0.669
207      1.325
2907     1.266
Name: Price, dtype: float64

11. Use the following help to translate the longitude and latitude of the data to find the cities corresponding to each observation: [geopy](https://pypi.org/project/geopy)

In [26]:
!pip install geopy

In [27]:
# Example of how to get the adress from a given pair of latitude/longitude coordinates
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="yet_another_app")
location = geolocator.reverse("52.509669, 13.376294")
loc_dict = dict(location.raw)
loc_dict["address"]

{'house_number': '11',
 'road': 'Potsdamer Platz',
 'suburb': 'Tiergarten',
 'borough': 'Mitte',
 'city': 'Berlin',
 'ISO3166-2-lvl4': 'DE-BE',
 'postcode': '10785',
 'country': 'Deutschland',
 'country_code': 'de'}

In [28]:
# Use geopy to extract the city of each row in the sample dataset
X_sample["City"] = 0
for i, row in X_sample.iterrows():
    geolocator = Nominatim(user_agent="yet_another_app_2")
    location = geolocator.reverse("{}, {}".format(X_sample.loc[i, "Latitude"], X_sample.loc[i, "Longitude"]), 
                                  timeout = None)
    loc_dict = dict(location.raw)
    try:
        X_sample.loc[i, "City"] = loc_dict["address"]["city"]
    except:
        try:
            X_sample.loc[i, "City"] = loc_dict["address"]["town"]
        except:
            try:
                X_sample.loc[i, "City"] = loc_dict["address"]["village"]
            except:
                pass
# If city was not found, replace by "Unknown"
X_sample.loc[X_sample['City'] == 0, 'City'] = "Unknown"

/var/folders/xs/9pw_49kx273dmym0qk_0l_s00000gn/T/ipykernel_28381/1970226461.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Sacramento' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_sample.loc[i, "City"] = loc_dict["address"]["city"]


In [29]:
X_sample.describe(include='all')

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_2,MedInc_3,...,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2,City
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,...,150.000000,1.500000e+02,150.000000,150.000000,150.000000,1.500000e+02,1.500000e+02,150.000000,150.000000,150
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unknown
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20
mean,3.856067,28.546667,5.216433,1.066535,1416.980000,2.854573,35.829133,-119.699000,17.895234,99.282082,...,46455.303323,1.681097e+06,0.028003,0.000787,14331.344402,-1.716284e+06,2.055878e+08,-0.008356,0.000070,NaN
std,1.745362,13.144313,1.220520,0.118717,904.890564,0.718236,2.073048,1.875433,18.587063,205.005072,...,8088.901998,3.901623e+05,0.001607,0.000090,449.134872,8.068462e+04,1.288632e+07,0.000131,0.000002,NaN
min,1.125000,3.000000,2.059524,0.880342,76.000000,1.443754,32.680000,-122.940000,1.265625,1.423828,...,34901.664832,1.140586e+06,0.024808,0.000615,13135.452100,-1.858145e+06,1.725401e+08,-0.008725,0.000066,NaN
25%,2.546150,18.250000,4.283824,1.009332,839.500000,2.315671,33.942500,-121.487500,6.482910,16.506612,...,39104.928957,1.327319e+06,0.026462,0.000700,13948.791100,-1.793060e+06,1.945688e+08,-0.008467,0.000068,NaN
50%,3.657800,29.000000,5.269823,1.056229,1221.000000,2.788445,35.720000,-119.280000,13.379700,48.941720,...,45587.474972,1.628801e+06,0.027998,0.000784,14227.718500,-1.697082e+06,2.024280e+08,-0.008384,0.000070,NaN
75%,4.868775,37.750000,6.081688,1.093684,1695.250000,3.216339,37.790000,-118.105000,23.706547,115.437122,...,53967.298139,2.039424e+06,0.029462,0.000868,14759.212675,-1.647422e+06,2.178344e+08,-0.008231,0.000072,NaN


12. Make a train/test splitting from X_sample and Y_sample

In [30]:
X_train, X_test, Y_train, Y_test = train_test_split(X_sample, Y_sample, test_size=0.2, random_state=42)

13. What preprocessings are necessary now ? The cells below implement the preprocessings, read it carefully and check what is done

In [31]:
categorical_features = ['City']
numeric_features = [c for c in X_sample.columns if c != 'City']

In [32]:
# Create transformer for numeric features
numeric_transformer = StandardScaler()

In [33]:
# Create transformer for categorical features
categorical_transformer = OneHotEncoder(drop='first', handle_unknown = 'ignore') # ignore if unknown categories are found in test set

In [34]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [35]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
X_train = preprocessor.fit_transform(X_train)
print('...Done.')

# Preprocessings on test set
print("Performing preprocessings on test set...")
X_test = preprocessor.transform(X_test) # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print('...Done.')

Performing preprocessings on train set...
...Done.
Performing preprocessings on test set...
...Done.


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


14. Train a regression model and evaluate the performances. Are you satisfied?

In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, Y_train)

print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

R2 score on training set :  0.9992293854026966
R2 score on test set :  -463594739.5631267
